# 03 - Optional denoising and sharpening experiment

Test filters only after comparing the original PDF, source image and rendered TIFF. Denoising can erase a thin minus sign; sharpening can strengthen noise. Neither is enabled automatically in an ingestion pipeline here.

This notebook is self-contained: supply an approved local TIFF/PNG/JPEG, preferably the controlled render you want to test, and follow [setup instructions](../README.md). It does not import notebook 02 or depend on its variables. To use its output, explicitly set `INPUT_IMAGE` to that TIFF path.

The experiment keeps all pages and compares three variants: a grayscale/lossless control, denoising alone, and sharpening alone. The shared grayscale control helps separate filter effects from re-encoding. No filter result is described as correct without checking the affected values.

In [ ]:
import hashlib, json, os
from pathlib import Path

import cv2 as cv
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter, ImageSequence
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.identity import DefaultAzureCredential
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
INPUT_IMAGE = Path('../input/fax-rendered-300dpi.tiff')
ANALYZER_ID = 'prebuilt-layout'
API_VERSION = '2025-11-01'
KEYWORD = 'property tax'
DENOISE_STRENGTH = 3
OUT = Path('../output/03-filters')
OUT.mkdir(parents=True, exist_ok=True)
print('Input:', INPUT_IMAGE, '| bytes:', INPUT_IMAGE.stat().st_size)

## Use existing filter implementations - local only
The denoising call uses OpenCV's documented [single-grayscale non-local-means API](https://docs.opencv.org/4.x/d5/d69/tutorial_py_non_local_means.html), described in the [pinned tutorial source](https://github.com/opencv/opencv/blob/55ff746075718ec2d7cfe09b106c070555f3dbff/doc/py_tutorials/py_photo/py_non_local_means/py_non_local_means.markdown). Sharpening uses Pillow's [UnsharpMask](https://pillow.readthedocs.io/en/stable/reference/ImageFilter.html#PIL.ImageFilter.UnsharpMask), with the [existing implementation](https://github.com/python-pillow/Pillow/blob/98fa02d46db2bc96f04521e5989d08a8b6083c9f/src/PIL/ImageFilter.py). These are library calls, not newly written filter algorithms.

Parameters are light starting points to test, not validated settings for your fax stream. All pages are converted to 8-bit grayscale and saved with lossless LZW, following the [existing TIFF-save pattern](https://github.com/jomedinagomez/azure-ai-foundry-insurance/blob/f6949572c5aaa6bc77b1ac592a92affc62d98346/demo/sov/preprocess/pdf_render.py). Source DPI is retained when present; no missing resolution is invented. This cell does not rescale pixels, deskew, binarize, erode or dilate.

In [ ]:
with Image.open(INPUT_IMAGE) as source:
    dpi = source.info.get('dpi')
    frames = [page.convert('L') for page in ImageSequence.Iterator(source)]
save_options = {'dpi': dpi} if dpi is not None else {}
if dpi is None:
    print('No source DPI metadata: outputs will not invent a DPI value.')
variants = {
    'grayscale_control': frames,
    'denoised': [Image.fromarray(cv.fastNlMeansDenoising(
        np.asarray(page), None, h=DENOISE_STRENGTH, templateWindowSize=7, searchWindowSize=21
    )) for page in frames],
    'sharpened': [page.filter(ImageFilter.UnsharpMask(radius=1, percent=100, threshold=3)) for page in frames],
}
inputs = {}
try:
    for label, pages in variants.items():
        path = OUT / f'{label}.tiff'
        pages[0].save(path, format='TIFF', save_all=True, append_images=pages[1:],
                      compression='tiff_lzw', **save_options)
        inputs[label] = path
finally:
    for pages in variants.values():
        for page in pages:
            page.close()
pd.DataFrame([{'case': label, 'bytes': path.stat().st_size} for label, path in inputs.items()])

## Compare the three variants - Azure calls
The request and polling flow is adapted from Microsoft's [binary-analysis sample](https://github.com/Azure/azure-sdk-for-python/blob/53023a2cd9adf255cff520a99985fff84dc58d7d/sdk/contentunderstanding/azure-ai-contentunderstanding/samples/sample_analyze_binary.py), with local output files and the same OCR/layout analyzer for every variant. Microsoft copyright and MIT terms: [LICENSE](../LICENSE).

The setup cell creates the client. The comparison cell below submits three fresh, billable analyses on each execution. No custom analyzers or model deployments are created.

In [ ]:
endpoint = os.getenv('CONTENTUNDERSTANDING_ENDPOINT') or os.environ['APP_CONTENT_UNDERSTANDING_ENDPOINT']
client = ContentUnderstandingClient(
    endpoint=endpoint,
    credential=DefaultAzureCredential(process_timeout=60),
    api_version=API_VERSION,
)

def analyze_file(path, label):
    file_bytes = path.read_bytes()
    poller = client.begin_analyze_binary(analyzer_id=ANALYZER_ID, binary_input=file_bytes)
    result = poller.result()
    markdown = '\n\n'.join(content.markdown or '' for content in result.contents)
    (OUT / f'{label}.json').write_text(json.dumps(result.as_dict(), indent=2), encoding='utf-8')
    (OUT / f'{label}.md').write_text(markdown, encoding='utf-8')
    evidence = {'case': label, 'file': path.name, 'sha256': hashlib.sha256(file_bytes).hexdigest(),
                'analyzer': ANALYZER_ID, 'api_version': API_VERSION}
    (OUT / f'{label}.input.json').write_text(json.dumps(evidence, indent=2), encoding='utf-8')
    return markdown

In [ ]:
texts = {}
for label, path in inputs.items():
    print('Analyzing', label, 'with', ANALYZER_ID)
    texts[label] = analyze_file(path, label)

## Inspect before choosing a variant
Compare the affected amount and sign with the image, and check nearby values for regressions. If a filter removes a visible sign, reject it even if other text looks cleaner. Full Markdown and JSON are saved locally; the preview below is intentionally small.

Only consider a filter after representative failing and non-failing documents improve without unacceptable regressions. Keep the original file, parameters, hashes and result JSON when preparing an approved reproduction. Rendering or image-level redaction may remove the symptom; confirm that a shared reproduction still exhibits the failure before treating it as evidence.

In [ ]:
for label, text in texts.items():
    print('\n---', label, '---')
    lines = text.splitlines()
    matches = [i for i, line in enumerate(lines) if KEYWORD.casefold() in line.casefold()]
    if not matches:
        print('Label not found. Inspect the complete saved Markdown; no accuracy conclusion was made.')
    for i in matches[:3]:
        print('\n'.join(lines[max(0, i - 1):i + 3]))